# Recomendación colaborativa de películas: Evaluación

Ordenar películas para usuarios existentes de una muestra de MovieLens 32M.

Hit rate y NDCG@10 sobre un objetivo positivo retenido por usuario. Los objetivos de ítems sin historial cuentan como fallos. Corte por usuario, sin corte global; no evalúa nuevos usuarios.

Consulta la [metodología](../docs/metodologia.md). Los reportes corresponden a la última ejecución de cada etapa.


In [1]:
from pathlib import Path
import os
import sys
from IPython.display import Markdown, display
def _buscar_proyecto(paquete):
    """Encuentra el paquete propio desde el proyecto, tema o portafolio."""
    for base in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (base / "src" / paquete).is_dir():
            return base
        for container in (base, base / "Analisis"):
            for pattern in (f"*/src/{paquete}", f"*/*/src/{paquete}"):
                for source in sorted(container.glob(pattern)):
                    parts = source.relative_to(container).parts
                    if "back" in parts or any(p.startswith(".") for p in parts):
                        continue
                    candidate = source.parent.parent
                    if (candidate / "pyproject.toml").is_file():
                        return candidate
    return None

PROJECT_ROOT = _buscar_proyecto("movielens_recommender")
if PROJECT_ROOT is None:
    raise RuntimeError("Abre el notebook desde el proyecto, notebooks o su carpeta contenedora.")
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))
for folder in ("data/raw", "data/intermediate", "data/processed", "data/output", "models", "reports"):
    (PROJECT_ROOT / folder).mkdir(parents=True, exist_ok=True)
from movielens_recommender.alerts import configurar_notebook, mostrar_alertas
configurar_notebook()


In [2]:
from movielens_recommender.flow import run_stage
metricas, comentario = run_stage(PROJECT_ROOT, "evaluacion")
display(Markdown(comentario))
from movielens_recommender.presentation import display_evaluation
display_evaluation(PROJECT_ROOT, metricas)


# Recomendación colaborativa de películas: evaluacion

El modelo supera el baseline por NDCG@10: **0.0448** frente a **0.0253**, en **469 objetivos positivos**. Hay **19 objetivos sin historial del ítem** en train.

Se evalúa una muestra de usuarios existentes con un objetivo retenido por usuario; no mide satisfacción real ni usuarios nuevos. Los cortes son temporales por usuario, no un corte global: las interacciones de otros usuarios pueden ser posteriores al objetivo de un usuario. Un top solicitado puede quedar incompleto si no hay suficientes candidatos con señal positiva.


,baseline,modelo
usuarios_con_objetivo_positivo,469.000000,469.000000
objetivos_sin_historial_item,19.000000,19.000000
hit_rate_at_k,0.046908,0.085288
ndcg_at_k,0.025255,0.044848
recomendaciones_promedio,10.000000,10.000000
cobertura_items_train,0.009105,0.025941
k,10.000000,10.000000


Usuario del ejemplo: 983


,movieId,title
0,58559,"Dark Knight, The (2008)"
1,260,Star Wars: Episode IV - A New Hope (1977)
2,593,"Silence of the Lambs, The (1991)"
3,59315,Iron Man (2008)
4,33794,Batman Begins (2005)
5,589,Terminator 2: Judgment Day (1991)
6,527,Schindler's List (1993)
7,4011,Snatch (2000)
8,109487,Interstellar (2014)
9,1210,Star Wars: Episode VI - Return of the Jedi (1983)


## Alertas

Avisos agrupados con descripción y acción sugerida. Los errores no se ocultan; no registrar warnings no garantiza que el análisis sea válido.

In [ ]:
mostrar_alertas(PROJECT_ROOT / "reports/alertas/05_evaluacion.json")
from movielens_recommender.informe import generar_informe
generar_informe(PROJECT_ROOT)
